# 24 — Target networks: a copy that lags

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/024-dqn.ipynb)

Q-learning moves each value toward a target, and that target is built from the very values being moved. A target network breaks the loop by bootstrapping from an old copy of the weights. You will train the same toy with and without one, drag the refresh interval, and then pick a step size where the difference stops being cosmetic.

## 1. The numbers: one state, one target

The toy is a five-state corridor: every step moves right and pays a small reward, and state 4 is terminal. The model is a straight line `w0 + w1*s/3`, so two weights have to describe all five values.

In [ ]:
import numpy as np                                   # the array library
gamma = 0.99                                         # future reward counts almost as much as now
rewards = np.array([1.03, 1.02, 1.01, 1.00, 0.0])    # paid on leaving each state; state 4 is terminal
values = np.array([4., 3., 2., 1., 0.])              # the true value of each state
w = np.zeros(2)                                      # the model: w[0] + w[1] * s / 3
feat = lambda s: np.array([1., s / 3.])              # the two features of state s
print("value of state 2 now:", feat(2) @ w)          # 0.0 before any update
target = rewards[2] + gamma * (feat(3) @ w)          # the target: reward plus the discounted next value
print("target for state 2:", round(target, 3))       # 1.01 + 0.99 * 0.0
print("TD error:", round(target - feat(2) @ w, 3))   # what the update will shrink

**Line by line**

- `import numpy as np` — loads the array library.
- `gamma = 0.99` — the discount on future reward.
- `rewards = ...` — the reward paid on leaving each state.
- `values = ...` — the true value, known here only so we can measure the error.
- `w = np.zeros(2)` — the whole model, two numbers.
- `feat = lambda s: ...` — a function turning a state into its two features.
- `print("value of state 2 now:", ...)` — the prediction before learning.
- `target = ...` — reward plus the discounted value of the next state.
- `print("target ...")` — one step of lookahead knows nothing about the far end yet.
- `print("TD error ...")` — the gap the update closes.

## 2. Step by step: one update at a time

Each step picks a state, builds a target from the frozen copy, and nudges the weights toward it. The copy is refreshed every `sync` steps; a large `sync` means a long-frozen target. Here `sync = 10`, and the printed error is measured across all five states.

In [ ]:
def train(lr, sync, steps=500):                       # one run; returns the error after each step
    rng = np.random.default_rng(0)                   # fixed seed, so runs are comparable
    w = np.zeros(2)                                  # the online weights
    wbar = w.copy()                                  # the frozen copy used for targets
    errs = []                                        # one error per update
    for t in range(steps):                           # the loop over updates
        s = int(rng.integers(0, 4))                  # a state the agent visits
        nxt = s + 1                                  # its only action: move right
        boot = 0. if nxt >= 4 else feat(nxt) @ wbar  # state 4 is terminal, so it contributes zero
        y = rewards[s] + gamma * boot                # the Bellman target
        w = w - lr * ((feat(s) @ w) - y) * feat(s)   # one gradient step on the TD error
        if t % sync == 0:                            # every sync steps
            wbar = w.copy()                          # refresh the frozen copy
        pred = np.array([feat(k) @ w for k in range(5)])   # the model's value for all five states
        errs.append(float(np.abs(pred - values).max()))    # worst error in the table
    return np.array(errs)                            # the curve
e10 = train(0.5, 10)                                 # lr 0.5, target refreshed every 10 steps
for t in [0, 50, 150, 299, 499]:                     # five checkpoints
    print(f"step {t:3d}  worst error {e10[t]:.5f}")  # watch it fall

**Line by line**

- `def train(lr, sync, steps=500):` — the whole experiment in one function.
- `rng = np.random.default_rng(0)` — a seeded random source.
- `w = np.zeros(2)` — the online weights start at zero.
- `wbar = w.copy()` — a copy for building targets.
- `errs = []` — the error curve.
- `for t in range(steps):` — one update per iteration.
- `s = int(rng.integers(0, 4))` — pick a non-terminal state.
- `nxt = s + 1` — the successor.
- `boot = ...` — the frozen copy's value of the successor, or zero past the end.
- `y = ...` — reward plus discounted bootstrap.
- `w = w - lr * ...` — move the weights toward the target.
- `if t % sync == 0:` — the freezing schedule.
- `wbar = w.copy()` — refresh the copy.
- `pred = ...` — value of every state under the current weights.
- `errs.append(...)` — the worst of the five errors.
- `return np.array(errs)` — hand the curve back.
- `e10 = train(0.5, 10)` — run it once with a ten-step refresh.
- `for t in [...]:` — a few checkpoints.
- `print(...)` — error at each checkpoint.

## 3. Plot: with a target network and without

`sync = 1` refreshes the copy after every update, which is the same as having no target network at all. `sync = 10` keeps the target still for ten steps, so the weights move in short staircases.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library
e1 = train(0.5, 1)                                   # no freeze: the target is rebuilt every step
plt.figure(figsize=(6, 3))                           # one wide panel
plt.plot(e1, label="sync = 1 (no target network)")   # the curve that chases itself
plt.plot(e10, label="sync = 10 (frozen copy)")       # the curve from section 2
plt.yscale("log")                                    # log y keeps both ends visible
plt.xlabel("update number")                          # x-axis label
plt.ylabel("worst value error (log)")                # y-axis label
plt.legend()                                         # name the two curves
plt.title("the frozen copy climbs in stairs and arrives a little later")   # takeaway
plt.show()                                           # render

**Line by line**

- `import matplotlib.pyplot as plt` — loads the plotting library.
- `e1 = train(0.5, 1)` — the same run with the target refreshed every step.
- `plt.figure(...)` — one panel.
- `plt.plot(e1, ...)` — the unfrozen curve.
- `plt.plot(e10, ...)` — the frozen curve from section 2.
- `plt.yscale("log")` — errors span several orders of magnitude.
- `plt.xlabel("update number")` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend()` — say which curve is which.
- `plt.title(...)` — the sentence the figure is making.
- `plt.show()` — render.

## 4. Interactive: how long to freeze

The slider is the refresh interval. Small values keep the target honest but jumpy; large values give clean staircases that start late. The title reports the error at the last update.

In [ ]:
from ipywidgets import interact                       # sliders that rerun a function
def show(sync=10):                                   # sync comes from the slider
    e = train(0.5, sync)                             # one full run at this interval
    plt.figure(figsize=(5, 3))                       # a fresh figure per drag
    plt.plot(e, color="tab:orange")                  # the error curve
    plt.yscale("log")                                # log y
    plt.xlabel("update number")                      # x-axis label
    plt.ylabel("worst value error (log)")            # y-axis label
    plt.title(f"sync = {sync}: final error {e[-1]:.1e}")   # live readout
    plt.show()                                       # draw it
    print(f"sync {sync:2d} -> final error {e[-1]:.2e}")    # the same number as text
interact(show, sync=(1, 80, 1))                      # the slider

**Line by line**

- `from ipywidgets import interact` — the slider widget.
- `def show(sync=10):` — one function per slider position.
- `e = train(0.5, sync)` — rerun training at this interval.
- `plt.figure(...)` — a new figure per drag.
- `plt.plot(e, ...)` — the error curve.
- `plt.yscale("log")` — log y.
- `plt.xlabel("update number")` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.title(...)` — the final error, printed on the plot.
- `plt.show()` — draw it.
- `print(...)` — the same number as text.
- `interact(show, sync=(1, 80, 1))` — the slider, its range and its step.

## 5. When it breaks: the target that follows the weights

Keep the data and the update, and raise the step size to 0.9. With a frozen copy the run still lands on the answer. With a target rebuilt from the same weights every step, each update moves the goalposts, the next target is bigger, and the error climbs instead of falling.

In [ ]:
bad = train(0.9, 1)                                  # big step, target rebuilt every step
good = train(0.9, 10)                                # big step, but the target is frozen
plt.figure(figsize=(6, 3))                           # one panel
plt.plot(bad, label="sync = 1 (no freeze)")          # the run that diverges
plt.plot(good, label="sync = 10 (frozen copy)")      # the run that still converges
plt.yscale("log")                                    # log y: 1e12 and 1e-6 on one axis
plt.xlabel("update number")                          # x-axis label
plt.ylabel("worst value error (log)")                # y-axis label
plt.legend()                                         # name the two curves
plt.title("an unfrozen target feeds its own error back")   # takeaway
plt.show()                                           # render
print(f"no freeze ends at {bad[-1]:.1e}, frozen at {good[-1]:.1e}")   # orders of magnitude apart

**Line by line**

- `bad = train(0.9, 1)` — step size 0.9, no freezing.
- `good = train(0.9, 10)` — same step size, ten-step freeze.
- `plt.figure(...)` — one panel.
- `plt.plot(bad, ...)` — the diverging curve.
- `plt.plot(good, ...)` — the converging curve.
- `plt.yscale("log")` — the only way to see both runs at once.
- `plt.xlabel("update number")` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend()` — name the curves.
- `plt.title(...)` — the sentence the figure is making.
- `plt.show()` — render.
- `print(...)` — the two final errors, side by side.

## Your turn

Predict the largest `sync` in section 4 that still ends below an error of 0.01, then check it with the slider. In section 5, find a step size where the frozen run is also unstable, and explain what the freeze can and cannot protect you from.